[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gregwchase/dynatrace-pubsec-summit-2026/blob/main/notebooks/05_roi.ipynb)

# Lab 5: ROI: Measure Before You Build, Prove It After
**Traces to Insights: A Hands-On Workshop with Arize AX** · Dynatrace Public Sector Summit 2026 · ⏱️ ~15 min

**In this lab you will:**
- Compute **cost per outcome**, not just cost per call, from your real traces and evals
- Model **time to value** and the payback period for an agent project
- Use a **measure-first worksheet** to decide what to measure before you commit resources

In [ ]:
# --- Setup: works in Google Colab or a local Jupyter environment ---
import os, sys, subprocess

if "google.colab" in sys.modules:
    REPO_DIR = "/content/dynatrace-pubsec-summit-2026"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/gregwchase/dynatrace-pubsec-summit-2026.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif not os.path.exists("civic_agent") and os.path.exists("../civic_agent"):
    os.chdir("..")  # running locally from notebooks/

sys.path.insert(0, os.getcwd())
from workshop_utils import load_credentials

PROJECT = load_credentials()  # prompts for keys if they aren't in .env / Colab Secrets

## 1. Cost per trace is the wrong headline

Cost per LLM call is easy to measure and easy to misread. A cheap agent that's wrong 30% of the time can cost more than an expensive one that's right 95% of the time, because every wrong answer becomes a phone call, a re-application, or a complaint.

**Cost per successful outcome** = (AI cost + cost of handling failures) ÷ number of requests resolved correctly

Let's compute it from your production traces.

In [ ]:
import pandas as pd
from workshop_utils import export_spans, root_spans, cost_per_trace

spans = export_spans(hours=24)
traces = root_spans(spans).merge(cost_per_trace(spans), on="context.trace_id", how="left")

# "Resolved" = passed every eval logged on that trace in Labs 2 and 4.
# Guardrail-blocked requests count as resolved; blocking an SSN is the correct outcome.
pass_cols = [c for c in ["eval.cites_source.score", "eval.eligibility_tool_used.score", "eval.hallucination.score"] if c in traces]
print("Using evals:", pass_cols)
scores = traces[pass_cols].apply(pd.to_numeric, errors="coerce") if pass_cols else pd.DataFrame(index=traces.index)
blocked = traces["guardrail_triggered"].fillna("none") != "none"
traces["resolved"] = blocked | (scores.notna().any(axis=1) & scores.min(axis=1, skipna=True).eq(1))
traces["cost_usd"] = traces["cost_usd"].fillna(0)

by_version = traces.groupby("prompt_version").agg(
    requests=("context.trace_id", "count"),
    resolved_rate=("resolved", "mean"),
    ai_cost_per_request=("cost_usd", "mean"),
)
by_version

## 2. Add the cost of failure

Every unresolved AI interaction has a cost: a resident calls the contact center, visits an office, or gives up. **Replace these assumptions with your agency's numbers.**

In [ ]:
# ---- Assumptions: change these! ----
COST_PER_HUMAN_CONTACT = 7.50      # fully loaded cost of a contact-center call (USD)
ESCALATION_RATE_ON_FAILURE = 0.60  # share of unresolved AI interactions that turn into a human contact
MONTHLY_REQUESTS = 40_000          # expected monthly volume for the service
# -------------------------------------

def unit_economics(resolved_rate, ai_cost):
    failure_cost = (1 - resolved_rate) * ESCALATION_RATE_ON_FAILURE * COST_PER_HUMAN_CONTACT
    total_per_request = ai_cost + failure_cost
    return pd.Series({
        "resolved_rate": resolved_rate,
        "ai_cost_per_request": ai_cost,
        "failure_cost_per_request": failure_cost,
        "cost_per_successful_outcome": total_per_request / max(resolved_rate, 1e-9),
        "monthly_total_cost": total_per_request * MONTHLY_REQUESTS,
        "monthly_savings_vs_all_human": (COST_PER_HUMAN_CONTACT - total_per_request) * MONTHLY_REQUESTS,
    })

econ = by_version.apply(lambda r: unit_economics(r.resolved_rate, r.ai_cost_per_request), axis=1)
econ.style.format({"resolved_rate": "{:.0%}", "ai_cost_per_request": "${:.4f}", "failure_cost_per_request": "${:.2f}",
                   "cost_per_successful_outcome": "${:.2f}", "monthly_total_cost": "${:,.0f}",
                   "monthly_savings_vs_all_human": "${:,.0f}"})

👀 Notice that the AI cost per request is a fraction of a cent, while the failure cost is measured in dollars. **Quality improvements usually beat model cost cuts for ROI.** That's why Labs 2 and 3 matter to the budget office, not only to engineers.

### Sensitivity: where does the business case break?

In [ ]:
import numpy as np

ai_cost = by_version["ai_cost_per_request"].mean()
grid = pd.DataFrame(
    {f"{r:.0%} resolved": [unit_economics(r, ai_cost * m)["monthly_savings_vs_all_human"] for m in (0.5, 1, 2, 5)]
     for r in (0.5, 0.7, 0.85, 0.95)},
    index=["0.5× model cost", "1× model cost", "2× model cost", "5× model cost"],
)
grid.style.format("${:,.0f}").background_gradient(cmap="RdYlGn", axis=None)

## 3. Time to value

ROI also depends on **how long it takes to get to a trustworthy resolved rate**. Without observability, teams spend weeks finding failures by hand. With traces, evals, and experiments, each iteration takes hours.

In [ ]:
BUILD_COST = 120_000          # one-time: engineering, security review, ATO work (USD)
ITERATION_WEEKS_MANUAL = 3    # weeks per improvement cycle without tracing/evals
ITERATION_WEEKS_WITH_AX = 0.5 # weeks per cycle with traces → evals → experiments
CYCLES_TO_TARGET = 4          # improvement cycles needed to reach your target resolved rate
target_savings = unit_economics(0.90, ai_cost)["monthly_savings_vs_all_human"]

for label, weeks in [("Manual review", ITERATION_WEEKS_MANUAL), ("With Arize AX loop", ITERATION_WEEKS_WITH_AX)]:
    weeks_to_target = weeks * CYCLES_TO_TARGET
    payback_months = weeks_to_target / 4.33 + BUILD_COST / max(target_savings, 1)
    print(f"{label:<20} time to target quality: {weeks_to_target:>4.1f} weeks · payback ≈ {payback_months:.1f} months")

## 4. Measure before you build: the worksheet

Fill this in **before** committing resources to an agent project. If you can't fill in a row, that's your first task.

| # | Question | Example (CivicAssist) | Yours |
|---|---|---|---|
| 1 | **What outcome** are we buying? (not "deploy an agent") | Residents get a correct benefits answer without calling | |
| 2 | **What's the baseline** today, and how do we know? | 40k calls/mo, $7.50/call, 6-min avg handle time | |
| 3 | **How will we measure "correct"?** Which evals, and against what golden dataset? | Correctness + hallucination + tool accuracy; 200-question SME golden set | |
| 4 | **What's the minimum quality bar** to launch? And to scale? | ≥ 90% correctness, 0 hallucinated eligibility rules on golden set | |
| 5 | **What's the cost guardrail?** | ≤ $0.02 AI cost per request, p95 latency < 8s | |
| 6 | **What's the cost of a failure**, and who bears it? | Escalated call ($7.50), or a resident wrongly told they're ineligible (equity risk) | |
| 7 | **What's the kill / pivot criterion?** | < 75% resolved after 6 iteration cycles | |
| 8 | **Who sees the dashboard**, and how often? | Program director weekly; security officer for guardrail trends | |

🤔 **Discussion prompts:**
- Which of these can you measure **today** with your current systems?
- In the public sector, what "returns" don't show up in dollars? (resident trust, equity of access, staff burnout, after-hours availability)
- How would you present *cost per outcome* to a CFO or a program director instead of *cost per token*?

---

## 🎉 You did the full loop

**Observe** (traces) → **Measure** (evals) → **Improve** (datasets, Playground, experiments) → **Catch issues in production** (Signal, online evals, dashboards, monitors) → **Prove value** (ROI)

Take this repo with you. Point `setup_tracing()` at your own agent, reuse the evaluator patterns, and fill in the worksheet with your team.